# Data Downsampling

Downsample quality-controlled data using configuration-driven parameters.

**Configuration Source**: Uses pipeline parameters if available, otherwise uses defaults from `downsampling_config.json`

In [ ]:
import sys
from pathlib import Path
import warnings
import pandas as pd
import json
import os
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path('../src').resolve()))
from downsampling import DataDownsampler

# Load downsampling configuration
config_dir = Path('../config')
with open(config_dir / 'downsampling_config.json', 'r') as f:
    downsampling_config = json.load(f)

## Configuration

In [ ]:
# Load configuration
config_dir = Path('../config')
with open(config_dir / 'downsampling_config.json', 'r') as f:
    config = json.load(f)

# Use configuration settings
frequency = config['settings']['frequency']

# Use pipeline output directory if running from main pipeline
import os
if 'PIPELINE_OUTPUT_ROOT' in os.environ:
    OUTPUT_ROOT = Path(os.environ['PIPELINE_OUTPUT_ROOT'])
    input_dir = OUTPUT_ROOT / 'quality_controlled'
    output_dir = OUTPUT_ROOT / 'downsampled'
else:
    input_dir = Path(config['input_directory'])
    output_dir = Path(config['output_directory'])

print(f"Configuration:")
print(f"  Input: {input_dir}")
print(f"  Output: {output_dir}")
print(f"  Frequency: {frequency}")

# Validate input directory
if not input_dir.exists():
    print(f"WARNING: Input directory not found: {input_dir}")

# Create output directories
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'microclimate').mkdir(exist_ok=True)
(output_dir / 'power').mkdir(exist_ok=True)
(output_dir / 'weather').mkdir(exist_ok=True)

# Initialize downsampler with config settings
downsampler = DataDownsampler(default_frequency=frequency)

# Microclimate Data Downsampling

In [ ]:
# Process microclimate data
microclimate_files = list((input_dir / 'microclimate').glob('*.csv'))

if not microclimate_files:
    print("WARNING: No microclimate files found")
else:
    for file in sorted(microclimate_files):
        output_file = output_dir / 'microclimate' / file.name
        print(f"  {file.name}")
        downsampler.downsample_qc_merged_file(file, output_file, frequency=frequency)    
        print(f"✓ Microclimate downsampling complete: {len(microclimate_files)} files")

# Power Data Downsampling

In [ ]:
# Process power data
power_files = list((input_dir / 'power').glob('*.csv'))

if not power_files:
    print("WARNING: No power files found")
else:
    for file in sorted(power_files):
        output_file = output_dir / 'power' / file.name
        print(f"  {file.name}")    
        print(f"✓ Power downsampling complete: {len(power_files)} files")
        downsampler.downsample_qc_merged_file(file, output_file, frequency=frequency)

# Weather Data Downsampling

In [ ]:
# Process weather data
weather_files = list((input_dir / 'weather').glob('*.csv'))

if not weather_files:
    print("WARNING: No weather files found")
else:
    for file in sorted(weather_files):
        output_file = output_dir / 'weather' / file.name
        print(f"  {file.name}")
        downsampler.downsample_qc_merged_file(file, output_file, frequency=frequency)
    print(f"✓ Weather downsampling complete: {len(weather_files)} files")